In [1]:
#Import the petting zoo environment and evaluate
from Multi_Ag_Environment import CustomEnvironment
from pettingzoo.test import parallel_api_test
import gymnasium as gym
env = CustomEnvironment()
parallel_api_test(env, num_cycles=1_000_000)

[51, 36, 8]
New Matrix
[[0.04000427456187727, 10, ['Satellite3']], [0.12151544061087027, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.6362258149770601, 10, ['Satellite3', 'Satellite1']], [0.9738626719007847, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.7294782556219731, 10, ['Satellite2', 'Satellite1', 'Satellite3']], [0.7970907798794039, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.39967829959821677, 10, ['Satellite3']], [0.6970954731980493, 10, ['Satellite3', 'Satellite1', 'Satellite2']], [0.2679339019364678, 10, ['Satellite1', 'Satellite3', 'Satellite2']], [0.7078307889949411, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.4778118608118016, 10, ['Satellite2', 'Satellite1']], [0.4533769957136943, 10, ['Satellite1']], [0.9556912457089897, 10, ['Satellite2']], [0.09440542299295251, 10, ['Satellite2']], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -

In [2]:
#Heuristic Baselines
#CGR
#More intelligent heuristic
import random

# Initialize the env
env = CustomEnvironment()
observations = env.reset()

# Use a fixed agent list
all_agents = ["satellite1", "satellite2", "satellite3"]

# Track done flags
terminated = {agent: False for agent in all_agents}
truncated = {agent: False for agent in all_agents}

while not all([terminated[a] or truncated[a] for a in all_agents]):
    actions = {
        agent: env.action_space(agent).sample()
        for agent in all_agents
        if not (terminated[agent] or truncated[agent])
    }
    print("Agent Actions")
    print(actions)
    observations, rewards, terminated, truncated, infos = env.step(actions)
    print(rewards)
env.close()


[13, 83, 51]
New Matrix
[[0.8667078733754329, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.8284441098337724, 10, ['Satellite2', 'Satellite1']], [0.7730876644433337, 10, ['Satellite1', 'Satellite3', 'Satellite2']], [0.8471294847601003, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.5016220192811938, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.6592721752122803, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.971777023811146, 10, ['Satellite1']], [0.4719576566105985, 10, ['Satellite2', 'Satellite1']], [0.09671618038385887, 10, ['Satellite1', 'Satellite3']], [0.2116210393558543, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.8473322009717855, 10, ['Satellite3']], [0.013509742879781017, 10, ['Satellite3']], [0.949078224949552, 10, ['Satellite2']], [0.8603443159924825, 10, ['Satellite2', 'Satellite3']], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1,

In [ ]:
import os

import ray
import supersuit as ss
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.tune.registry import register_env
from torch import nn

#from pettingzoo.butterfly import pistonball_v6


class CNNModelV2(TorchModelV2, nn.Module):
    def __init__(self, obs_space, act_space, num_outputs, *args, **kwargs):
        TorchModelV2.__init__(self, obs_space, act_space, num_outputs, *args, **kwargs)
        nn.Module.__init__(self)
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, [8, 8], stride=(4, 4)),
            nn.ReLU(),
            nn.Conv2d(32, 64, [4, 4], stride=(2, 2)),
            nn.ReLU(),
            nn.Conv2d(64, 64, [3, 3], stride=(1, 1)),
            nn.ReLU(),
            nn.Flatten(),
            (nn.Linear(3136, 512)),
            nn.ReLU(),
        )
        self.policy_fn = nn.Linear(512, num_outputs)
        self.value_fn = nn.Linear(512, 1)

    def forward(self, input_dict, state, seq_lens):
        model_out = self.model(input_dict["obs"].permute(0, 3, 1, 2))
        self._value_out = self.value_fn(model_out)
        return self.policy_fn(model_out), state

    def value_function(self):
        return self._value_out.flatten()


def env_creator(args):
    env = CustomEnvironment()
    #env = ss.color_reduction_v0(env, mode="B")
    #env = ss.dtype_v0(env, "float32")
    #env = ss.resize_v1(env, x_size=84, y_size=84)
    #env = ss.normalize_obs_v0(env, env_min=0, env_max=1)
    #env = ss.frame_stack_v1(env, 3)
    return env


if __name__ == "__main__":
    ray.init()

    env_name = "CustomEnvironment"

    register_env(env_name, lambda config: ParallelPettingZooEnv(env_creator(config)))
    ModelCatalog.register_custom_model("CNNModelV2", CNNModelV2)

    config = (
    PPOConfig()
    .environment(env=env_name, clip_actions=True)
    .env_runners(num_env_runners=4, rollout_fragment_length=128)
    .training(
        train_batch_size=512,
        lr=2e-5,
        gamma=0.99,
        lambda_=0.9,
        use_gae=True,
        clip_param=0.4,
        grad_clip=None,
        entropy_coeff=0.1,
        vf_loss_coeff=0.25,
    )
    .framework("torch")
    .debugging(log_level="ERROR")
    .resources(num_gpus=int(os.environ.get("RLLIB_NUM_GPUS", "0")))
    .update_from_dict({
        "num_sgd_iter": 10,
        "sgd_minibatch_size": 64,  # ✅ correct place
    })
)

    tune.run(
        "PPO",
        name="PPO",
        stop={"timesteps_total": 5000000 if not os.environ.get("CI") else 50000},
        checkpoint_freq=10,
        storage_path="~/ray_results/" + env_name,
        config=config.to_dict(),
    )